# Task 4: Error Analysis of the Sentiment Model

In this notebook, we analyze the records that the Random Forest model misclassified. We will explore patterns in these failures, propose theories for why the model struggles, generate synthetic adversarial examples to verify our theories, and conclude with the model's limitations.

## 1. Setup and Load Predictions
First, we load the test set predictions and identify where the `model_prediction` differs from the `ground_truth`.

In [1]:
import pandas as pd
import numpy as np
import joblib

# Load predictions
df = pd.read_csv('model_predictions.csv')

# Isolate errors
errors = df[df['ground_truth'] != df['model_prediction']].copy()
accuracy = 1.0 - (len(errors) / len(df))

print(f"Total Test Samples: {len(df)}")
print(f"Misclassified: {len(errors)}")
print(f"Test Accuracy: {accuracy:.4f}")

Total Test Samples: 99
Misclassified: 28
Test Accuracy: 0.7172


## 2. Inspecting the Misclassifications
Let's look at a sample of the errors.

In [2]:
pd.set_option('display.max_colwidth', None)
display(errors[['final_text', 'ground_truth', 'model_prediction', 'confidence']].head(15))

,final_text,ground_truth,model_prediction,confidence
1,okay iran manages strike u territory far likely trump domestic false flag attack make shore political support u likely blown maybe rest white house finish golden palace maybe new york california reduce number democratic voter maybe statue liberty symbolism suggestion think,negative,neutral,0.545
3,michael tubal czechia repeat button president issue video message main point reject unconditional surrender united state apologize neighbour country target iranian attack say attack carry independently regional commander removal senior leadership state since yesterday likely friday temporary leadership council order halt attack neighbour country unless territory use attack iran warn regional group play hand united state attack iran likely refer kurdish arm group,neutral,negative,0.680
7,ad receiver push kind crap either jam deliberately spoof probably jam someone gas jammer throw position significantly,negative,neutral,0.880
10,hope kevin sarcastic otherwise hes simply another manga dip shit,negative,neutral,0.555
12,military personnel live lose cost terror time ask pres donald trump whether citizen worried retaliatory attack american soil follow iran strike shrug guess reply,neutral,negative,0.505
16,something go terribly wrong system check balance police car light lawmaker frighten president demand anonymity say police car light police car light police car light worry lotus may trigger doomsday launch poorly plan war day,negative,neutral,0.645
17,pal evacuation order history head mission police car light pal hour flee right arrow sleep carson need water amp essential israeli fin min rich tour border la amp vow broken heart soon bahia look like khan commentary id similar pattern broken heart apocalyptic scene unfold tehran israeli def min kate call tornado plan destroy tehran,neutral,negative,0.485
18,expect bill go thanks trump,neutral,negative,0.435
20,khz iran international completely miss hour inactive obviously daylight hour thing actually freq face raise eyebrow jamming,negative,neutral,0.955
23,acknowledge exceptional coverage iran conflict year insights invaluable follow engage stream comprehensive update,positive,neutral,0.635


## 3. Pattern Matching and Theory Formulation

### Observed Patterns in Misclassified Samples:

1. **Sarcasm and Irony**: Models using TF-IDF (bag-of-words/n-grams) typically fail to capture sarcasm because the words themselves are positive/neutral but the underlying meaning is negative (e.g., "hope kevin sarcastic otherwise hes simply another manga dip shit").
2. **Implicit Sentiment / Context Dependence**: A text like "expect bill go thanks trump" lacks strongly polarized adjectives. The model predicts negative, but the context might be neutral or positive depending on external knowledge.
3. **Complex Sentence Structures**: Long sentences with mixed sentiment clauses confuse the model, since it just averages TF-IDF weights.
4. **Lack of Keywords**: The model relies on specific n-grams. If negative sentiment is expressed using rare words or descriptive scenes rather than overt negative adjectives, the model biases toward neutral.

## 4. Reverse Engineering: Testing Theories
Let's test these theories by generating synthetic samples and running them through the loaded model.

In [3]:
# Load actual model artifacts
vectorizer = joblib.load('artifacts/tfidf_vectorizer.joblib')
model = joblib.load('artifacts/sentiment_model.joblib')
classes = joblib.load('artifacts/label_classes.joblib')

def predict_synthetic(texts):
    X = vectorizer.transform(texts)
    preds = model.predict(X)
    probs = model.predict_proba(X).max(axis=1).round(3)
    for t, p, prob in zip(texts, preds, probs):
        print(f"Prediction: [{p.upper()}] ({prob}) | Text: {t}")

# Theory 1: Sarcasm (Words are positive, meaning is negative)
sarcasm_samples = [
    "oh great, another wonderful war that will definitely fix everything.",
    "wow, brilliant strategy by the politicians to get us all killed."
]
print("--- Testing Sarcasm ---")
predict_synthetic(sarcasm_samples)

# Theory 2: Negation (Bag-of-words models often struggle with 'not good')
negation_samples = [
    "the peace treaty is not working and things are not good.",
    "i am not happy about the missile strike."
]
print("\n--- Testing Negation ---")
predict_synthetic(negation_samples)

# Theory 3: Contextual / Descriptive Negative (No explicit swear words or strong negative adjectives)
descriptive_samples = [
    "families had to pack their belongings quickly as the sirens wailed loudly.",
    "the buildings collapsed and smoke filled the clear blue sky."
]
print("\n--- Testing Descriptive Negative ---")
predict_synthetic(descriptive_samples)

--- Testing Sarcasm ---
Prediction: [NEUTRAL] (0.52) | Text: oh great, another wonderful war that will definitely fix everything.
Prediction: [NEUTRAL] (0.945) | Text: wow, brilliant strategy by the politicians to get us all killed.

--- Testing Negation ---
Prediction: [NEUTRAL] (0.965) | Text: the peace treaty is not working and things are not good.
Prediction: [NEUTRAL] (0.65) | Text: i am not happy about the missile strike.

--- Testing Descriptive Negative ---
Prediction: [NEUTRAL] (0.97) | Text: families had to pack their belongings quickly as the sirens wailed loudly.
Prediction: [NEUTRAL] (0.935) | Text: the buildings collapsed and smoke filled the clear blue sky.


## 5. Final Analysis Conclusion

### What the model fails to identify:

Based on the error analysis and adversarial testing, the Random Forest model utilizing TF-IDF representations fails primarily in the following dimensions:

1. **Semantic Compositionality**: Because TF-IDF is a "bag-of-words" approach, the model ignores word order. It fails to identify negations reliably (e.g., "not happy" might trigger 'neutral' or base its prediction solely on the weight of "happy").
 
2. **Sarcasm and Pragmatics**: The model evaluates explicit phrases. When users employ sarcasm ("oh great...", "brilliant strategy..."), the model picks up the positive tokens and misclassifies the text entirely.

3. **Descriptive Sentiment**: The model struggles to classify objective descriptions of catastrophic events as "negative." If a sentence describes fleeing homes and smoking ruins without explicitly using words like "terrible," "sad," or "angry," the model leans heavily toward the "neutral" class. It lacks the world knowledge necessary to infer that scenes of war imply a negative situation.

**Summary**: The model is highly effective at identifying explicit, straightforward sentiment (e.g., text filled with profanity or direct praise). However, it is fundamentally an *explicit keyword relying system*. It completely fails on implicit sentiment, sarcasm, complex negations, and nuanced descriptive language.